# 04A — Standalone Improved Student: Architecture & Training

**Purpose:** Train the Improved Student model **without knowledge distillation** — pure supervised learning with cross-entropy + class weights.

## Architecture

`10 × 30 s PSG epochs → Lite Multi-Resolution Stem → Depthwise-Separable CNN → Parametric Gabor FEB → 2-Layer GRU → 5-Class classifier`

This is the **Student-only** experiment (Notebook 4A). The Teacher–Student distillation experiment is in Notebook 4B.

### Configuration
- Sequence length: **10 epochs** (300 s context)
- Input channels: **4** | Classes: **5**
- Training budget: **20 epochs**
- Loss: **Focal Loss** (imbalance-aware)
- Checkpoint: `artifacts/student_standalone_best.pt`


In [ ]:
import math
import time
import random
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from sklearn.metrics import cohen_kappa_score, accuracy_score, f1_score
SEED = 42
SEQ_LEN = 10
SEQ_STRIDE = 5
EPOCHS = 20
BATCH_SIZE = 16
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
N_CHANNELS = 4
N_CLASSES = 5
FS = 100
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PROJECT_ROOT = Path("/home/shamique/projects/sleep")
CACHE_DIR = PROJECT_ROOT / "data/cache"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
RESULTS_DIR = PROJECT_ROOT / "results"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
STUDENT_CHECKPOINT = ARTIFACT_DIR / "student_standalone_best.pt"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Device:", DEVICE)
print("Student checkpoint:", STUDENT_CHECKPOINT)

## 1. Sequence dataset

Each training example contains ten contiguous 30-second epochs. Windows are formed independently inside each subject-night cache so no temporal sequence crosses recording boundaries, and the subject-level split is respected (no subject leakage).

In [ ]:
class SleepSequenceDataset(Dataset):
    def __init__(self, cache_index_df, split, seq_len=SEQ_LEN, stride=SEQ_STRIDE):
        self.samples = []
        self.cache = {}
        self.seq_len = seq_len

        rows = cache_index_df.loc[cache_index_df["split"] == split]

        for _, row in rows.iterrows():
            path = row["cache_path"]
            data = np.load(path)
            n = len(data["labels"])

            for start in range(0, n - seq_len + 1, stride):
                self.samples.append((path, start))

    def _load(self, path):
        if path not in self.cache:
            d = np.load(path)
            self.cache[path] = (d["epochs"], d["labels"])
        return self.cache[path]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, start = self.samples[index]
        epochs, labels = self._load(path)

        end = min(start + self.seq_len, len(labels))
        x = epochs[start:end]
        y = labels[start:end]

        if len(x) < self.seq_len:
            pad = self.seq_len - len(x)
            x = np.concatenate([x, np.repeat(x[-1:], pad, axis=0)], axis=0)
            y = np.concatenate([y, np.repeat(y[-1:], pad, axis=0)], axis=0)

        # Verify temporal continuity using orig_epoch_idx if available
        data = np.load(path)
        if "orig_epoch_idx" in data:
            orig_idx = data["orig_epoch_idx"]
            window_idx = orig_idx[start:end]
            if len(window_idx) > 1 and not np.all(np.diff(window_idx) == 1):
                raise ValueError(f"Temporal gap in window for {path} at start={start}")

        return torch.from_numpy(x).float(), torch.from_numpy(y).long()


cache_index = pd.read_csv(CACHE_DIR / "cache_index.csv")

import json
# Use exhibition manifest as single source of split truth
with open('../data/manifests/exhibition_15subj_v1.json') as f:
    exhibition_manifest = json.load(f)
train_subjects = exhibition_manifest['train_subjects']
val_subjects = exhibition_manifest['validation_subjects']
test_subjects = exhibition_manifest['test_subjects']

# Filter cache index by manifest subjects
train_cache = cache_index[cache_index['subject_id'].isin(train_subjects)]
val_cache = cache_index[cache_index['subject_id'].isin(val_subjects)]
test_cache = cache_index[cache_index['subject_id'].isin(test_subjects)]

# Verify no overlap
assert set(train_subjects).isdisjoint(val_subjects)
assert set(train_subjects).isdisjoint(test_subjects)
assert set(val_subjects).isdisjoint(test_subjects)

train_ds = SleepSequenceDataset(train_cache, "train")
val_ds = SleepSequenceDataset(val_cache, "val")
test_ds = SleepSequenceDataset(test_cache, "test")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Sequence windows:")
print("train:", len(train_ds))
print("val:  ", len(val_ds))
print("test: ", len(test_ds))

## 2. Model architecture

**Improved Student (standalone):** Lite multi-resolution stem → depthwise-separable CNN → parametric Gabor FEB → 2-layer GRU.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from sleep_staging.models import ImprovedStudent
student = ImprovedStudent().to(DEVICE)
n_student = sum(p.numel() for p in student.parameters())
print(f"Improved Student parameters: {n_student:,}  (standalone model)")
with torch.no_grad():
    smoke = torch.randn(2, SEQ_LEN, N_CHANNELS, 30 * FS, device=DEVICE)
    s_logits, s_feat = student(smoke, return_features=True)
print("Student logits:", tuple(s_logits.shape), "| features:", tuple(s_feat.shape))

## 3. Class weighting from the training partition

Class weights are computed from training labels only. The validation and test partitions are never used to determine training weights.

In [ ]:
train_labels = []

for _, row in cache_index.loc[cache_index["split"] == "train"].iterrows():
    train_labels.append(np.load(row["cache_path"])["labels"])

train_labels = np.concatenate(train_labels)
class_counts = np.bincount(train_labels, minlength=N_CLASSES)

counts = torch.tensor(class_counts, dtype=torch.float32, device=DEVICE)
class_weights = torch.log(counts.sum() / (counts + 1.0))
class_weights = class_weights / class_weights.mean()

print("Training class counts:", class_counts.tolist())
print("Class weights:", [round(float(w), 3) for w in class_weights])

## 4. Training objective — Focal Loss (no distillation)

**Focal loss** handles class imbalance without requiring a teacher model.

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, weight, gamma=1.5):
        super().__init__()
        self.w = weight
        self.gamma = gamma

    def forward(self, logits, labels):
        C = logits.shape[-1]
        ce = F.cross_entropy(
            logits.reshape(-1, C), labels.reshape(-1),
            weight=self.w.to(logits.device), reduction="none",
        )
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()


focal_loss = FocalLoss(class_weights)
print("FocalLoss ready (no distillation).")

## 5. Training & evaluation utilities (per-epoch logging)

Every training loop logs **every epoch** — train loss, validation Cohen's κ, validation accuracy, and macro-F1 — and checkpoints only the best model according to validation κ.

In [ ]:
@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    y_true, y_pred = [], []
    for x, y in loader:
        logits = model(x.to(DEVICE))
        pred = logits.argmax(dim=-1).cpu().numpy()
        y_true.append(y.numpy().reshape(-1))
        y_pred.append(pred.reshape(-1))
    return np.concatenate(y_true), np.concatenate(y_pred)


def evaluate(model, loader, tag, epoch, total_epochs, train_loss):
    y_true, y_pred = collect_predictions(model, loader)
    kappa = cohen_kappa_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    mf1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    print(
        f"  [{tag}] Epoch {epoch:02d}/{total_epochs:02d} | "
        f"train_loss={train_loss:.4f} | val_kappa={kappa:.4f} | "
        f"val_acc={acc:.4f} | val_macroF1={mf1:.4f}"
    )
    return {"kappa": kappa, "acc": acc, "mf1": mf1}


def cosine_warmup(optimizer, total_steps):
    def lr_lambda(s):
        if s < 0.1 * total_steps:
            return s / (0.1 * total_steps)
        return 0.5 * (1 + math.cos(math.pi * (s - 0.1 * total_steps) / (0.9 * total_steps)))
    return LambdaLR(optimizer, lr_lambda)


def train_model(model, loss_fn, train_ld, val_ld, epochs, tag, ckpt_path,
                train_subjects=None,
                val_subjects=None,
                test_subjects=None):
    opt = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    sched = cosine_warmup(opt, epochs * len(train_ld))
    best_kappa = -1.0
    history = []

    for ep in range(epochs):
        model.train()
        loss_sum, n_batches = 0.0, 0
        for x, y in train_ld:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(x), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            sched.step()
            loss_sum += float(loss.detach())
            n_batches += 1

        metrics = evaluate(
            model, val_ld, tag, ep + 1, epochs,
            loss_sum / max(n_batches, 1),
        )
        history.append({"epoch": ep + 1, "train_loss": loss_sum / max(n_batches, 1), **metrics})

        if metrics["kappa"] > best_kappa:
            best_kappa = metrics["kappa"]
            checkpoint = {
                "model_state_dict": model.state_dict(),
                "experiment_id": "EXP-STUDENT-STANDALONE",
                "dataset": "Sleep-EDF Expanded",
                "n_records": len(train_subjects) + len(val_subjects) + len(test_subjects),
                "n_persons": len(set(train_subjects + val_subjects + test_subjects)),
                "split_manifest": "data/manifests/exhibition_15subj_v1.json",
                "split_type": "subject",
                "seed": SEED,
                "sequence_length": SEQ_LEN,
                "training_stride": SEQ_STRIDE,
                "evaluation_protocol": "legacy_all_position",
                "train_subjects": train_subjects,
                "validation_subjects": val_subjects,
                "test_subjects": test_subjects,
                "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip() if Path(".git").exists() else "unknown",
                "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
            }
            torch.save(checkpoint, ckpt_path)
            print(f"    -> new best {tag} (kappa={best_kappa:.4f}); checkpoint saved.")

    payload = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    if "model_state_dict" in payload:
        state = payload["model_state_dict"]
    else:
        state = payload
    model.load_state_dict(state)
    return model, pd.DataFrame(history)

print("Training utilities ready (standalone, no teacher).")

## 6. Train the Improved Student (standalone, per-epoch logging)

The student is trained from scratch on the training split with focal loss. Each epoch prints its full validation metrics; only the best-κ checkpoint is kept.

**No teacher, no distillation — pure supervised learning.**

In [ ]:
print("Training Improved Student (standalone)")
print("=" * 80)
t0 = time.time()

student, student_history = train_model(
    student, focal_loss, train_loader, val_loader, EPOCHS,
    "student_standalone", STUDENT_CHECKPOINT,
    train_subjects=train_subjects,
    val_subjects=val_subjects,
    test_subjects=test_subjects,
)

print(f"Student training finished in {time.time() - t0:.1f}s | best val kappa: {student_history['kappa'].max():.4f}")
display(student_history[['epoch', 'train_loss', 'kappa', 'acc', 'mf1']])

## 7. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(student_history["epoch"], student_history["train_loss"], label="Student (standalone)", marker="o")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Train loss")
axes[0].set_title("Training loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(student_history["epoch"], student_history["kappa"], label="Student (standalone)", marker="s", color="orange")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Validation Cohen's κ")
axes[1].set_title("Validation κ per epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Handoff

`artifacts/student_standalone_best.pt` is the standalone student checkpoint trained **without distillation**.

This is **Notebook 4A** — the standalone Improved Student experiment.
**Notebook 4B** contains the Teacher–Student distillation experiment.

Both checkpoints can be evaluated in Notebook 05 for comparison.